In [1]:
import pandas as pd
import re

In [9]:
# Load dataset
df = pd.read_csv("../storage/geniuslyrics.csv", low_memory=False,encoding='utf-8-sig')
print("Initial shape:", df.shape)

Initial shape: (6678, 13)


In [ ]:
# Function to fix brackets that span multiple lines
def fix_multiline_brackets(text):
    if not isinstance(text, str):
        return text
    
    # Replace newlines inside square brackets with spaces
    fixed_text = re.sub(
        r'\[(.*?)\]',
        lambda match: '[' + re.sub(r'\s*\n\s*', ' ', match.group(1)).strip() + ']',
        text,
        flags=re.DOTALL
    )
    
    return fixed_text

df['genius_lyrics'] = df['genius_lyrics'].apply(fix_multiline_brackets)

In [4]:
# Function to split lyrics into verses
def split_lyrics_sections(lyrics):
    if not isinstance(lyrics, str):
        return []
    
    # Split based on [Section ...]
    parts = re.split(r'\[(.*?)\]', lyrics)
    
    sections = []
    
    # parts structure: [text_before, label1, text1, label2, text2, ...]
    for i in range(1, len(parts), 2):
        section_name = parts[i]
        section_text = parts[i+1].strip()
        
        if section_text:  # avoid empty sections
            sections.append((section_name, section_text))
    
    return sections

new_rows = []

In [5]:
new_rows = []

# Iterate through each row and split lyrics into sections
for _, row in df.iterrows():
    sections = split_lyrics_sections(row['genius_lyrics'])
    
    for section_name, section_text in sections:
        new_row = row.to_dict()
        new_row['section'] = section_name
        new_row['genius_lyrics'] = section_text
        
        new_rows.append(new_row)

In [10]:
# Create new dataframe
verse_df = pd.DataFrame(new_rows)

# Format verses into a single line (separated by commas)
verse_df['genius_lyrics'] = verse_df['genius_lyrics'].apply(lambda x: ', '.join(x.splitlines()))

# Insert a new column 'verse_id' at the first position
verse_df.insert(0, 'verse_id', range(1, len(verse_df) + 1))

# Remove 'lyrics' columns
verse_df.drop(columns=['lyrics'], inplace=True, errors='ignore')

# Save output
print("After splitting:", verse_df.shape)
verse_df.to_csv("../storage/splitverses.csv", index=False, encoding='utf-8-sig')

After splitting: (51815, 14)
